# Agent 정의

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal


In [8]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [9]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [10]:
openai = OpenAI()

In [11]:
# Some lists!

todos = []
completed = []

In [12]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [13]:
get_todo_report()

''

In [14]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [15]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [16]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [17]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [18]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [19]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [20]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [21]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [22]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [23]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [24]:
todos, completed = [], []
loop(messages)

Todo #1: Interpret problem and set up relative motion timeline (account for 1-hour head start).
Todo #2: Estimate/assume Boston–New York distance since not provided; use a reasonable standard rail/road distance 
approximation.
Todo #3: Solve for meeting time using relative speed and starting separation after head start.
Todo #4: Present meeting time in clock time with clear reasoning.

Set 2:00 pm as t=0. Boston train speed 60 mph. New York train starts at 3:00 pm = t=1 hour, speed 80 mph, heading 
toward Boston. Need meeting time after t=0.

Todo #1: Interpret problem and set up relative motion timeline (account for 1-hour head start).
Todo #2: Estimate/assume Boston–New York distance since not provided; use a reasonable standard rail/road distance 
approximation.
Todo #3: Solve for meeting time using relative speed and starting separation after head start.
Todo #4: Present meeting time in clock time with clear reasoning.

Distance not given; assume a common approximate Boston–NYC distance of about 215 miles (typical driving/rail order 
of magnitude). Use D ≈ 215 mi.

Todo #1: Interpret problem and set up relative motion timeline (account for 1-hour head start).
Todo #2: Estimate/assume Boston–New York distance since not provided; use a reasonable standard rail/road distance 
approximation.
Todo #3: Solve for meeting time using relative speed and starting separation after head start.
Todo #4: Present meeting time in clock time with clear reasoning.

By 3:00 pm, Boston train has gone 60 mi, so remaining separation = D−60 = 215−60 = 155 mi. From 3:00 pm onward, 
closing speed = 60+80 = 140 mph. Time after 3:00 pm to meet = 155/140 = 1.1071 h ≈ 1 h 6 min 26 s.

Todo #1: Interpret problem and set up relative motion timeline (account for 1-hour head start).
Todo #2: Estimate/assume Boston–New York distance since not provided; use a reasonable standard rail/road distance 
approximation.
Todo #3: Solve for meeting time using relative speed and starting separation after head start.
Todo #4: Present meeting time in clock time with clear reasoning.

Meeting time = 3:00 pm + 1:06:26 ≈ 4:06 pm (about 4:06:30 pm).

Todo #1: Interpret problem and set up relative motion timeline (account for 1-hour head start).
Todo #2: Estimate/assume Boston–New York distance since not provided; use a reasonable standard rail/road distance 
approximation.
Todo #3: Solve for meeting time using relative speed and starting separation after head start.
Todo #4: Present meeting time in clock time with clear reasoning.

They meet at about 4:06 pm (≈ 4:06:30 pm), assuming Boston–New York is about 215 miles apart.

• By 3:00 pm, the Boston train has a 1-hour head start: 60 mph × 1 hr = 60 miles.  
• Remaining distance at 3:00 pm: 215 − 60 = 155 miles.  
• From 3:00 pm on, they close at 60 + 80 = 140 mph.  
• Time to meet after 3:00 pm: 155/140 hr ≈ 1.107 hr ≈ 1 hr 6 min.  
• 3:00 pm + 1 hr 6 min ≈ 4:06 pm.